# romacoustics BRAS Validation

Scenes 08 (CR1 classroom) and 09 (CR2 seminar) from the BRAS benchmark.
Compares computed T30 at 125 and 250 Hz against measured RIRs.

Also validates: eigenfrequencies (2D analytical), arbitrary geometry (L-shaped), ROM accuracy.

In [ ]:
import subprocess, sys, os
if not os.path.exists('repo'):
    subprocess.run(['git','clone','--depth','1','https://github.com/Burhanuddin98/Reduced-Order-Modelling-SL.git','repo'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','repo/romacoustics'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','gmsh'],check=True)
import numpy as np, time
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt, find_peaks
from romacoustics import Room
print('romacoustics imported')

In [ ]:
# Shared utilities
def compute_t30(sig,sr):
    e=sig.astype(np.float64)**2;edc=np.cumsum(e[::-1])[::-1];edc/=edc[0]+1e-30
    edc_db=10*np.log10(edc+1e-30);t=np.arange(len(edc_db))/sr
    i0=np.searchsorted(-edc_db,5);i1=np.searchsorted(-edc_db,35)
    if i1<=i0+2 or i1>=len(edc_db):return 0.0,0.0
    c=np.polyfit(t[i0:i1],edc_db[i0:i1],1)
    if c[0]>=0:return 0.0,0.0
    rt=-60.0/c[0];f=np.polyval(c,t[i0:i1])
    r2=1-np.sum((edc_db[i0:i1]-f)**2)/(np.sum((edc_db[i0:i1]-edc_db[i0:i1].mean())**2)+1e-30)
    return max(rt,0),r2

def band_t30(sig,sr,fc):
    nyq=sr/2;fl=fc/np.sqrt(2);fh=min(fc*np.sqrt(2),nyq*0.95)
    sos=butter(4,[fl/nyq,fh/nyq],btype='band',output='sos')
    return compute_t30(sosfiltfilt(sos,sig),sr)

rho_c=1.225*343.0
def alpha_to_Z(a):
    a=max(a,0.001);R=np.sqrt(1-a)
    return rho_c*(1+R)/(1-R)

## Test 1: Eigenfrequencies (2D rigid rectangle)

In [ ]:
Lx,Ly,c=2.0,1.0,343.0
room=Room.box_2d(Lx,Ly,ne=20,order=4)
room.set_source(0.7,0.3,sigma=0.15);room.set_receiver(1.5,0.7)
room.set_boundary_fi(Zs=1e8)
ir=room.solve(t_max=0.2,f_max=1000,Ns=500)
fft_mag=np.abs(np.fft.rfft(ir.signal));fft_f=np.fft.rfftfreq(len(ir.signal),1/ir.fs)
analytical=sorted([c/2*np.sqrt((n/Lx)**2+(m/Ly)**2) for n in range(10) for m in range(10) if n+m>0 and c/2*np.sqrt((n/Lx)**2+(m/Ly)**2)<900])
peaks,_=find_peaks(fft_mag,height=np.max(fft_mag)*0.01,distance=int(5/(fft_f[1]-fft_f[0])))
pk=fft_f[peaks];matched=0
for fa in analytical[:15]:
    best=pk[np.argmin(np.abs(pk-fa))] if len(pk)>0 else 0
    err=abs(best-fa)
    if err<5:matched+=1
    print(f'{fa:8.1f}Hz -> {best:8.1f}Hz  err={err:.1f}Hz')
print(f'Matched: {matched}/15')

## Test 2: BRAS Scene 09 (CR2 seminar room, 8.4 x 6.7 x 3.0 m)
Measured T30: broadband=1.68s, 125Hz=1.45s, 250Hz=1.69s

In [ ]:
room09=Room.box_3d(8.4,6.7,3.0,ne=6,order=4)
print(f'Scene 09: {room09.N} DOFs')
room09.set_material('z_min',alpha_to_Z(0.091))
room09.set_material('z_max',alpha_to_Z(0.104))
room09.set_material('x_min',alpha_to_Z(0.073))
room09.set_material('x_max',alpha_to_Z(0.050))
room09.set_material('y_min',alpha_to_Z(0.075))
room09.set_material('y_max',alpha_to_Z(0.075))
room09.set_source(2.0,3.35,1.5);room09.set_receiver(6.0,2.0,1.2)
t0=time.perf_counter()
ir09=room09.solve(t_max=2.0,f_max=300,Ns=300)
print(f'Solve: {time.perf_counter()-t0:.0f}s')
meas09={'bb':1.677,125:1.453,250:1.692}
bb,_=compute_t30(ir09.signal,ir09.fs)
print(f'Broadband T30={bb:.3f}s (measured: {meas09["bb"]:.3f}s, error: {abs(bb-meas09["bb"])/meas09["bb"]*100:.1f}%)')
for fc in [125,250]:
    t30,r2=band_t30(ir09.signal,ir09.fs,fc);m=meas09[fc]
    print(f'{fc}Hz: T30={t30:.3f}s (measured: {m:.3f}s, error: {abs(t30-m)/m*100:.1f}%)')

## Test 3: BRAS Scene 08 (CR1 classroom, 12.4 x 9.3 x 2.8 m)
Measured T30: broadband=6.00s, 125Hz=6.80s, 250Hz=5.48s

In [ ]:
room08=Room.box_3d(12.4,9.3,2.8,ne=5,order=4)
print(f'Scene 08: {room08.N} DOFs')
room08.set_material('z_min',alpha_to_Z(0.067))
room08.set_material('z_max',alpha_to_Z(0.013))
room08.set_material('x_min',alpha_to_Z(0.067))
room08.set_material('x_max',alpha_to_Z(0.067))
room08.set_material('y_min',alpha_to_Z(0.013))
room08.set_material('y_max',alpha_to_Z(0.210))
room08.set_source(3.0,4.0,1.4);room08.set_receiver(9.0,3.0,1.2)
t0=time.perf_counter()
ir08=room08.solve(t_max=4.0,f_max=250,Ns=250)
print(f'Solve: {time.perf_counter()-t0:.0f}s')
meas08={'bb':6.002,125:6.801,250:5.479}
bb,_=compute_t30(ir08.signal,ir08.fs)
print(f'Broadband T30={bb:.3f}s (measured: {meas08["bb"]:.3f}s, error: {abs(bb-meas08["bb"])/meas08["bb"]*100:.1f}%)')
for fc in [125,250]:
    t30,r2=band_t30(ir08.signal,ir08.fs,fc);m=meas08[fc]
    print(f'{fc}Hz: T30={t30:.3f}s (measured: {m:.3f}s, error: {abs(t30-m)/m*100:.1f}%)')

## Test 4: Arbitrary geometry (L-shaped room via Gmsh)

In [ ]:
import tempfile
geo='lc=0.4;\nPoint(1)={0,0,0,lc};Point(2)={4,0,0,lc};Point(3)={4,3,0,lc};\nPoint(4)={2,3,0,lc};Point(5)={2,5,0,lc};Point(6)={0,5,0,lc};\nLine(1)={1,2};Line(2)={2,3};Line(3)={3,4};Line(4)={4,5};Line(5)={5,6};Line(6)={6,1};\nCurve Loop(1)={1,2,3,4,5,6};Plane Surface(1)={1};\nExtrude{0,0,2.5}{Surface{1};}\nPhysical Volume(1)={1};'
tmp=tempfile.NamedTemporaryFile(suffix='.geo',delete=False,mode='w');tmp.write(geo);tmp.close()
room_L=Room.from_gmsh(tmp.name,f_max=200)
vol=room_L.ops['M_diag'].sum()
print(f'DOFs: {room_L.N}, Surfaces: {room_L._surface_labels}')
print(f'Volume: {vol:.1f} m3 (expected: 40.0 m3)')
room_L.set_source(1,1,1.2);room_L.set_receiver(3,1,1)
room_L.set_material('floor','carpet_thick');room_L.set_material('ceiling','plaster')
t0=time.perf_counter()
ir_L=room_L.solve(t_max=0.3,f_max=200,Ns=150)
print(f'Solve: {time.perf_counter()-t0:.0f}s, T30={ir_L.T30:.3f}s, C80={ir_L.C80:.1f}dB')
os.unlink(tmp.name)

## Test 5: ROM vs FOM accuracy

In [ ]:
room_r=Room.box_2d(2,2,ne=15,order=4)
room_r.set_source(1,1,sigma=0.2);room_r.set_receiver(0.3,0.3);room_r.set_boundary_fi(Zs=5000)
ir_fom=room_r.solve(t_max=0.1)
rom=room_r.build_rom(Z_train=[500,5000,15500])
ir_rom=rom.solve(Zs=5000)
n=min(len(ir_fom.signal),len(ir_rom.signal))
err=np.linalg.norm(ir_fom.signal[:n]-ir_rom.signal[:n])/(np.linalg.norm(ir_fom.signal[:n])+1e-30)
print(f'ROM basis: {rom.Nrb}, trained error: {err:.2e}')
room_r.set_boundary_fi(Zs=3000);ir_fom3=room_r.solve(t_max=0.1);ir_rom3=rom.solve(Zs=3000)
n=min(len(ir_fom3.signal),len(ir_rom3.signal))
err3=np.linalg.norm(ir_fom3.signal[:n]-ir_rom3.signal[:n])/(np.linalg.norm(ir_fom3.signal[:n])+1e-30)
print(f'Untrained Z=3000: error={err3:.2e}, FOM T30={ir_fom3.T30:.4f}s, ROM T30={ir_rom3.T30:.4f}s')